# Exploración del Dataset — Producción de Pozos No Convencionales

Dataset: Producción de Pozos de Gas y Petróleo No Convencional (datos.gob.ar)  
Fuente: Secretaría de Energía de la Nación Argentina

Este notebook explora el dataset que vamos a usar como base para el Feature Store y el modelo de pronóstico.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_URL = (
    "http://datos.energia.gob.ar/dataset/c846e79c-026c-4040-897f-1ad3543b407c"
    "/resource/b5b58cdc-9e07-41f9-b392-fb9ec68b0725"
    "/download/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv"
)
RAW_PATH = "../data/raw/pozos.csv"

## 1. Carga del dataset

In [ ]:
from pathlib import Path

if Path(RAW_PATH).exists():
    print(f"Cargando desde caché: {RAW_PATH}")
    df = pd.read_csv(RAW_PATH)
else:
    print("Descargando desde datos.gob.ar...")
    df = pd.read_csv(DATA_URL)
    Path(RAW_PATH).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(RAW_PATH, index=False)
    print(f"Guardado en {RAW_PATH}")

df["fecha"] = pd.to_datetime(df["fecha_data"])
print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")

## 2. Estructura del dataset

In [ ]:
print("=== Tipos de datos ===")
print(df.dtypes.to_string())
print()
print("=== Primeras filas ===")
df.head(3)

## 3. Rango temporal y cobertura

In [ ]:
print(f"Fecha mínima:       {df['fecha'].min().date()}")
print(f"Fecha máxima:       {df['fecha'].max().date()}")
print(f"Pozos únicos:       {df['idpozo'].nunique():,}")
print(f"Total registros:    {len(df):,}")
print()
print("=== Distribución por provincia ===")
print(df["provincia"].value_counts().to_string())
print()
print("=== Distribución por cuenca ===")
print(df["cuenca"].value_counts().to_string())

## 4. Variables de producción

In [ ]:
print("=== Estadísticas de producción ===")
df[["prod_pet", "prod_gas", "prod_agua"]].describe()

In [ ]:
# Hay valores negativos mínimos (-0.001 en pet, -12.26 en gas) → se clampean a 0 en el pipeline
print("Registros con prod_pet < 0:", (df["prod_pet"] < 0).sum())
print("Registros con prod_gas < 0:", (df["prod_gas"] < 0).sum())
print("Registros con prod_agua < 0:", (df["prod_agua"] < 0).sum())
print()
print("Registros con prod_pet == 0:", (df["prod_pet"] == 0).sum(), f"({(df['prod_pet']==0).mean():.1%})")
print("Registros con prod_gas == 0:", (df["prod_gas"] == 0).sum(), f"({(df['prod_gas']==0).mean():.1%})")

## 5. Valores nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
nulos_df = pd.DataFrame({"nulos": nulos, "%": nulos_pct})
nulos_df[nulos_df["nulos"] > 0].sort_values("%", ascending=False)

## 6. Variables relevantes para el Feature Store

Las features que vamos a usar son: `prod_pet`, `prod_gas`, `prod_agua`, `profundidad`, `tipoextraccion`.
Acá exploramos su distribución y cobertura.

In [ ]:
print("=== tipoextraccion ===")
print(df["tipoextraccion"].value_counts().to_string())
print()
print("=== profundidad describe ===")
print(df["profundidad"].describe().to_string())
print()
# Profundidad: hay valores 0 y un max de 378939 que parecen outliers
print("Registros con profundidad == 0:", (df["profundidad"] == 0).sum())
print("Registros con profundidad > 10000:", (df["profundidad"] > 10000).sum())

## 7. Visualizaciones

### 7.1 Histogramas de producción (escala log para ver la distribución completa)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title, color in zip(
    axes,
    ["prod_pet", "prod_gas", "prod_agua"],
    ["Petróleo (m³/mes)", "Gas (Mm³/mes)", "Agua (m³/mes)"],
    ["steelblue", "tomato", "seagreen"],
):
    data = df[df[col] > 0][col]
    ax.hist(np.log1p(data), bins=60, color=color, edgecolor="white", linewidth=0.3)
    ax.set_title(title)
    ax.set_xlabel("log(1 + producción)")
    ax.set_ylabel("Frecuencia")

fig.suptitle("Distribución de producción (registros con valor > 0, escala log)", fontsize=13)
plt.tight_layout()
plt.show()
print(f"Nota: se excluyen los {(df['prod_pet']==0).sum():,} registros con producción cero.")

### 7.2 Producción total mensual agregada (toda la cuenca)

In [ ]:
monthly = df.groupby("fecha")[["prod_pet", "prod_gas"]].sum().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(monthly["fecha"], monthly["prod_pet"], color="steelblue", linewidth=1.5)
axes[0].set_title("Producción total de petróleo (m³/mes) — No Convencional Argentina")
axes[0].set_ylabel("m³/mes")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

axes[1].plot(monthly["fecha"], monthly["prod_gas"], color="tomato", linewidth=1.5)
axes[1].set_title("Producción total de gas (Mm³/mes) — No Convencional Argentina")
axes[1].set_ylabel("Mm³/mes")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

for ax in axes:
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### 7.3 Serie temporal de pozos individuales (muestra)

In [ ]:
# Elegimos los 6 pozos petroleros con mayor producción total para visualizar
top_pozos = (
    df.groupby("idpozo")["prod_pet"]
    .sum()
    .nlargest(6)
    .index
    .tolist()
)

fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey=False)

for ax, pozo_id in zip(axes.flat, top_pozos):
    pozo_df = df[df["idpozo"] == pozo_id].sort_values("fecha")
    ax.plot(pozo_df["fecha"], pozo_df["prod_pet"], color="steelblue", linewidth=1.2)
    ax.set_title(f"Pozo {pozo_id}", fontsize=10)
    ax.set_ylabel("prod_pet (m³)")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30, labelsize=8)

fig.suptitle("Producción de petróleo — top 6 pozos (curvas de declinación típicas)", fontsize=13)
plt.tight_layout()
plt.show()

### 7.4 Historial por pozo: ¿cuántos meses tiene cada uno?

In [ ]:
registros_por_pozo = df.groupby("idpozo").size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(registros_por_pozo, bins=50, color="mediumpurple", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Cantidad de registros mensuales por pozo")
ax.set_ylabel("Cantidad de pozos")
ax.set_title("Distribución del historial disponible por pozo")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mediana de registros por pozo: {registros_por_pozo.median():.0f} meses")
print(f"Pozos con >= 10 registros:     {(registros_por_pozo >= 10).sum():,} ({(registros_por_pozo >= 10).mean():.1%})")
print(f"Pozos con < 10 registros:      {(registros_por_pozo < 10).sum():,}")
print()
print("Nota: la feature 'avg_prod_pet_10m' usa ventana de 10 meses — los pozos con menos")
print("registros usarán min_periods=1, lo cual está contemplado en el pipeline.")

## 8. Conclusiones para el Feature Store

Resumen de decisiones que informan el diseño del feature pipeline.

In [ ]:
conclusiones = {
    "Dataset": f"{len(df):,} registros, {df['idpozo'].nunique():,} pozos únicos, 2006–2026",
    "Columna fecha": "fecha_data (formato YYYY-MM-DD, fin de mes) → renombrar a 'fecha'",
    "Target del modelo": "prod_pet (m³/mes) — distribución muy sesgada, con muchos ceros",
    "Valores negativos": "prod_pet y prod_gas tienen negativos mínimos → clampear a 0 en clean_data()",
    "Ceros": f"{(df['prod_pet']==0).mean():.1%} de registros tienen prod_pet=0 (pozos inactivos o sin extracción)",
    "profundidad": "Sin nulos, pero outlier máximo de 378939m → usar media del pozo para reemplazar valores extremos",
    "tipoextraccion": f"601 nulos ({601/len(df):.2%}) → rellenar con 'Sin Sistema de Extracción' o categoría separada",
    "Ventana rolling (10m)": "Suficiente: mediana de historial por pozo es alta; pozos jóvenes usan min_periods=1",
    "Columnas a descartar": "vida_util (97.9% nulos), observaciones (94.4% nulos), idusuario, rectificado, habilitado",
}

for k, v in conclusiones.items():
    print(f"• {k}:\n  {v}\n")